<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
SEED = 42
VALIDATION_SIZE = 0.20
DATA_SAMPLE_SIZE = 12000
MAX_ITER = 25
BATCH_SIZE = 256

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("assignment")

2026/05/18 07:17:39 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/ganto/Downloads/mlruns/325186821407843097', creation_time=1779099459506, experiment_id='325186821407843097', last_update_time=1779099459506, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

Resposta: Sim, é necessário normalizar. O MLP usa otimização por gradiente; quando os pixels ficam na escala original 0-255, os pesos recebem atualizações maiores e menos estáveis. Ao trazer os atributos para 0-1, o treinamento converge de forma mais previsível, evita saturação principalmente em `logistic` e `tanh`, e torna a comparação entre configurações mais justa.

In [4]:
def load_data(seed=SEED, sample_size=DATA_SAMPLE_SIZE):
    """Carrega Fashion MNIST, normaliza pixels e separa treino/validacao."""
    fashion = fetch_openml(
        "Fashion-MNIST",
        version=1,
        as_frame=False,
        parser="auto",
    )

    X = fashion.data.astype(np.float32) / 255.0
    y = fashion.target.astype(np.int64)

    if sample_size is not None and sample_size < X.shape[0]:
        _, X, _, y = train_test_split(
            X,
            y,
            test_size=sample_size,
            stratify=y,
            random_state=seed,
        )

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=VALIDATION_SIZE,
        stratify=y,
        random_state=seed,
    )

    return X_train, X_val, y_train, y_val


X_train, X_val, y_train, y_val = load_data(SEED)

print(X_train.shape, X_val.shape)
print(pd.Series(y_train).value_counts().sort_index())

(9600, 784) (2400, 784)
0    960
1    960
2    960
3    960
4    960
5    960
6    960
7    960
8    960
9    960
Name: count, dtype: int64


# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [5]:
def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=5,
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [6]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

Foram usadas métricas ponderadas (`average="weighted"`) porque o problema é multiclasse. Mesmo o Fashion MNIST sendo aproximadamente balanceado, essa escolha mantém a avaliação correta caso a amostragem ou a divisão introduza pequenas diferenças entre classes.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [7]:
experiment_results = []


def run_experiment(
    run_name,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
):
    start = time.time()
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(
            {
                "activation": activation,
                "hidden_layers": str(hidden_layers),
                "learning_rate": learning_rate,
                "max_iter": max_iter,
                "batch_size": batch_size,
                "seed": SEED,
                "sample_size": DATA_SAMPLE_SIZE,
            }
        )

        model = train_mlp(
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=SEED,
            max_iter=max_iter,
            batch_size=batch_size,
        )

        training_time = time.time() - start
        metrics = evaluate(model, X_val, y_val)
        train_accuracy = accuracy_score(y_train, model.predict(X_train))

        mlflow.log_metrics(
            {
                **metrics,
                "training_time": training_time,
                "train_accuracy": train_accuracy,
                "n_iter": model.n_iter_,
                "loss": model.loss_,
            }
        )

    row = {
        "run_name": run_name,
        "activation": activation,
        "hidden_layers": hidden_layers,
        "learning_rate": learning_rate,
        "max_iter": max_iter,
        "batch_size": batch_size,
        **metrics,
        "train_accuracy": train_accuracy,
        "training_time": training_time,
        "n_iter": model.n_iter_,
        "loss": model.loss_,
    }
    experiment_results.append(row)
    return row, model


baseline_result, baseline_model = run_experiment(
    "baseline_relu_64_lr_0.001",
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
)

pd.DataFrame([baseline_result])

,run_name,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,train_accuracy,training_time,n_iter,loss
0,baseline_relu_64_lr_0.001,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.579073,25,0.301977


# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [8]:
activation_results = []

for activation in ["logistic", "tanh", "relu"]:
    result, _ = run_experiment(
        run_name=f"activation_{activation}",
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
    )
    activation_results.append(result)

activation_df = pd.DataFrame(activation_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
activation_df

,run_name,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,train_accuracy,training_time,n_iter,loss
1,activation_tanh,tanh,"(64,)",0.001,25,256,0.862500,0.862082,0.862500,0.861568,0.893125,1.235212,25,0.296412
0,activation_logistic,logistic,"(64,)",0.001,25,256,0.857500,0.856823,0.857500,0.856635,0.861042,9.179313,25,0.400474
2,activation_relu,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.511383,25,0.301977


## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

Resposta: Nesta execução, `tanh` teve o melhor desempenho entre as ativações, com F1-score de aproximadamente 0.862. `relu` treinou rápido, mas ficou levemente abaixo em validação, enquanto `logistic` teve tempo de treinamento bem maior. Assim, a melhor convergência prática ficou com `tanh`, e a maior estabilidade também ficou entre `tanh` e `relu`, com vantagem para `tanh` pela métrica final. Houve diferença significativa principalmente no custo de treinamento da `logistic`.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [9]:
architecture_results = []

for hidden_layers in [(32,), (64,), (128, 64), (256, 128)]:
    result, _ = run_experiment(
        run_name=f"architecture_{hidden_layers}",
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
    )
    architecture_results.append(result)

architecture_df = pd.DataFrame(architecture_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
architecture_df

,run_name,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,train_accuracy,training_time,n_iter,loss
3,"architecture_(256, 128)",relu,"(256, 128)",0.001,25,256,0.870000,0.871605,0.870000,0.869486,0.928021,5.368590,24,0.166983
2,"architecture_(128, 64)",relu,"(128, 64)",0.001,25,256,0.858750,0.862299,0.858750,0.856730,0.912083,2.966448,25,0.231841
1,"architecture_(64,)",relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.457500,25,0.301977
0,"architecture_(32,)",relu,"(32,)",0.001,25,256,0.855833,0.857439,0.855833,0.854330,0.879375,1.202852,25,0.356880


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

Resposta: Redes maiores não melhoraram os resultados de forma garantida, mas nesta execução a arquitetura `(256, 128)` obteve o melhor F1-score, cerca de 0.869. Ela também aumentou o tempo e a diferença entre acurácia de treino e validação, indicando maior risco de overfitting. O melhor tradeoff depende do critério: `(256, 128)` vence em desempenho final; `(128, 64)` é uma alternativa mais equilibrada quando custo e simplicidade pesam mais.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [10]:
learning_rate_results = []

for learning_rate in [0.1, 0.01, 0.001]:
    result, _ = run_experiment(
        run_name=f"learning_rate_{learning_rate}",
        activation="relu",
        hidden_layers=(64,),
        learning_rate=learning_rate,
    )
    learning_rate_results.append(result)

learning_rate_df = pd.DataFrame(learning_rate_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
learning_rate_df

,run_name,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,train_accuracy,training_time,n_iter,loss
2,learning_rate_0.001,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,8.129706,25,0.301977
1,learning_rate_0.01,relu,"(64,)",0.010,25,256,0.852917,0.853027,0.852917,0.850731,0.905729,1.403284,21,0.224916
0,learning_rate_0.1,relu,"(64,)",0.100,25,256,0.718750,0.743060,0.718750,0.685060,0.744479,1.052784,20,0.640246


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

Resposta: O learning rate `0.1` ficou instável e derrubou bastante as métricas de validação. O valor `0.01` convergiu rápido, mas não superou a configuração mais conservadora. Nesta execução, `0.001` apresentou o melhor comportamento geral, com maior F1-score e desempenho mais estável na validação.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


## Resposta final

- A ativação com melhor desempenho nesta execução foi `tanh`, com F1-score próximo de 0.862.
- A arquitetura com melhor resultado final foi `(256, 128)`, com F1-score próximo de 0.869; como tradeoff mais conservador, `(128, 64)` reduz custo e risco.
- O learning rate mais estável foi `0.001`; `0.1` foi o mais instável.
- Houve indício de overfitting nas arquiteturas maiores, principalmente porque a acurácia de treino ficou acima da acurácia de validação.
- A melhor configuração final registrada foi `relu` com arquitetura `(256, 128)` e learning rate `0.001`.
- As principais dificuldades foram controlar reprodutibilidade, manter o tempo de treinamento viável, comparar modelos de forma justa e registrar os experimentos de maneira rastreável no MLflow.

A célula abaixo consolida todos os resultados registrados durante a execução:

In [11]:
results_df = pd.DataFrame(experiment_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
results_df

,run_name,activation,hidden_layers,learning_rate,max_iter,batch_size,accuracy,precision,recall,f1_score,train_accuracy,training_time,n_iter,loss
7,"architecture_(256, 128)",relu,"(256, 128)",0.001,25,256,0.870000,0.871605,0.870000,0.869486,0.928021,5.368590,24,0.166983
2,activation_tanh,tanh,"(64,)",0.001,25,256,0.862500,0.862082,0.862500,0.861568,0.893125,1.235212,25,0.296412
6,"architecture_(128, 64)",relu,"(128, 64)",0.001,25,256,0.858750,0.862299,0.858750,0.856730,0.912083,2.966448,25,0.231841
1,activation_logistic,logistic,"(64,)",0.001,25,256,0.857500,0.856823,0.857500,0.856635,0.861042,9.179313,25,0.400474
0,baseline_relu_64_lr_0.001,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.579073,25,0.301977
3,activation_relu,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.511383,25,0.301977
5,"architecture_(64,)",relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,1.457500,25,0.301977
10,learning_rate_0.001,relu,"(64,)",0.001,25,256,0.857083,0.857528,0.857083,0.855425,0.891979,8.129706,25,0.301977
4,"architecture_(32,)",relu,"(32,)",0.001,25,256,0.855833,0.857439,0.855833,0.854330,0.879375,1.202852,25,0.356880
9,learning_rate_0.01,relu,"(64,)",0.010,25,256,0.852917,0.853027,0.852917,0.850731,0.905729,1.403284,21,0.224916
